In [1]:
import os
import glob
import pandas as pd

# Define the directory containing your CSV files
folder_path = "./hf_returns_daily"

# Find all CSV files in the folder
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

# This list will hold the processed DataFrames
all_dfs = []

for file_path in csv_files:
    # 1. Get the file name without the extension to use as the column name
    file_name = os.path.splitext(os.path.basename(file_path))[0]
    
    # 2. Read the CSV (assumes Date is column 0, Returns is column 1)
    # Modify 'names' if your columns have specific headers, or let pandas infer them
    df = pd.read_csv(file_path, index_col=0, parse_dates=True)
    
    # Ensure we only keep the return column (adjust if your column has a specific header)
    return_col = df.columns[0]
    df = df[[return_col]].copy()
    
    # 3. Rename the column to the fund's file name
    df.columns = [file_name]
    
    # 4. Normalize the date index to daily
    # This fixes issues where one file uses '2023-01-31' and another uses '2023-01-30'
    df.index = pd.to_datetime(df.index)
    
    # If a file has duplicate days for some reason, group and average them
    if df.index.duplicated().any():
        df = df.groupby(df.index).mean()
        
    all_dfs.append(df)

# 5. Concatenate all dataframes along the columns axis (outer join preserves all dates)
final_df = pd.concat(all_dfs, axis=1, join='outer')

# Sort chronologically
final_df = final_df.sort_index()
# View the final result
print(final_df.head())

               sp500  aqua_lake
date                           
2000-01-04 -0.038345        NaN
2000-01-05  0.001922        NaN
2000-01-06  0.000956        NaN
2000-01-07  0.027090        NaN
2000-01-10  0.011190        NaN


In [2]:
final_df.index = final_df.index.rename('date')

In [3]:
final_df.to_csv('concat_hf_data_daily.csv', index=True)